# Clase 213 — Kafka local + producer/consumer + snippets Kinesis

Notebook declarativo + simulación en proceso (no requiere Kafka). Para el demo real: levantar el `docker-compose.yml` de la celda 1.

## 1. docker-compose Kafka (KRaft, sin Zookeeper)

In [ ]:
compose = '''\
services:
  kafka:
    image: bitnami/kafka:3.7
    ports: ["9092:9092"]
    environment:
      KAFKA_CFG_NODE_ID: 1
      KAFKA_CFG_PROCESS_ROLES: controller,broker
      KAFKA_CFG_LISTENERS: PLAINTEXT://:9092,CONTROLLER://:9093
      KAFKA_CFG_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_CFG_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CFG_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_CFG_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_CFG_AUTO_CREATE_TOPICS_ENABLE: "true"
    healthcheck:
      test: kafka-topics.sh --bootstrap-server localhost:9092 --list
      interval: 5s

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    ports: ["8080:8080"]
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
    depends_on: { kafka: { condition: service_healthy } }
'''
print(compose)
print('# levantar: docker-compose up -d')
print('# UI: http://localhost:8080')

## 2. Producer Python

In [ ]:
producer_src = '''\
# producer.py — pip install confluent-kafka faker
from confluent_kafka import Producer
from faker import Faker
import json, time, random

fake = Faker()
p = Producer({
    "bootstrap.servers": "localhost:9092",
    "acks": "all",                  # waitall replicas
    "enable.idempotence": True,     # no dup en retries
    "linger.ms": 10,                # batchear hasta 10ms
})

def delivery(err, msg):
    if err: print(f"❌ {err}")

for i in range(1000):
    user_id = f"user_{random.randint(1, 100)}"
    event = {"page": fake.uri_path(), "ts": time.time(), "i": i}
    p.produce(
        "clicks",
        key=user_id.encode(),           # mismo user → misma partition
        value=json.dumps(event).encode(),
        callback=delivery,
    )
    if i % 100 == 0: p.poll(0)
p.flush(timeout=10)
print("1000 mensajes enviados.")
'''
print(producer_src)

## 3. Consumer Python (at-least-once con commit manual)

In [ ]:
consumer_src = '''\
# consumer.py
from confluent_kafka import Consumer, KafkaError
import json, duckdb

con = duckdb.connect("clicks.duckdb")
con.execute("""
    CREATE TABLE IF NOT EXISTS clicks (
        user_id TEXT, page TEXT, ts DOUBLE, i INT,
        PRIMARY KEY (user_id, i)          -- idempotent: dedupe a nivel de DB
    )
""")

c = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "clicks-to-duckdb",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,         # commit manual = at-least-once
})
c.subscribe(["clicks"])

try:
    while True:
        msg = c.poll(timeout=1.0)
        if msg is None: continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF: print(msg.error())
            continue
        user_id = msg.key().decode()
        event = json.loads(msg.value())
        con.execute("INSERT OR IGNORE INTO clicks VALUES (?, ?, ?, ?)",
                    [user_id, event["page"], event["ts"], event["i"]])
        c.commit(msg)   # commit DESPUÉS de procesar
finally:
    c.close()
    con.close()
'''
print(consumer_src)

## 4. Simulación in-process (sin Kafka real)

In [ ]:
from collections import defaultdict
import hashlib, json, time, random

class InMemoryTopic:
    def __init__(self, n_partitions=4):
        self.partitions = defaultdict(list)
        self.n = n_partitions
    def produce(self, key, value):
        p = int(hashlib.md5(key.encode()).hexdigest(), 16) % self.n
        self.partitions[p].append((key, value))
        return p

topic = InMemoryTopic(n_partitions=4)
for i in range(1000):
    key = f'user_{random.randint(1, 100)}'
    topic.produce(key, {'page': '/foo', 'i': i})

for p in range(topic.n):
    print(f'partition {p}: {len(topic.partitions[p]):>4} mensajes')

# Verificar: mismo user siempre va a la misma partition
u = 'user_42'
ps = set()
for _ in range(50):
    ps.add(topic.produce(u, {'i': 0}))
print(f'\nuser_42 fue a partitions: {ps} (debe ser solo 1)')

## 5. Kinesis equivalent

In [ ]:
kinesis = '''\
# Kinesis usa nombres distintos pero mismo modelo: shard ≈ partition.
import boto3, json
k = boto3.client("kinesis")

# Producer:
k.put_record(
    StreamName="clicks",
    Data=json.dumps({"page": "/foo"}).encode(),
    PartitionKey="user_42",   # → shard determinístico por hash de key
)

# Consumer enhanced fan-out (recommended):
shards = k.describe_stream(StreamName="clicks")["StreamDescription"]["Shards"]
for s in shards:
    iter_id = k.get_shard_iterator(
        StreamName="clicks", ShardId=s["ShardId"], ShardIteratorType="TRIM_HORIZON",
    )["ShardIterator"]
    out = k.get_records(ShardIterator=iter_id, Limit=10)
    for r in out["Records"]:
        print(json.loads(r["Data"]))

# Producción: usar KCL (Kinesis Client Library) — checkpoint en DynamoDB.
'''
print(kinesis)

## Ejercicio guiado

1. Levantá el docker-compose Kafka. Creá topic `clicks` con `--partitions 4`.
2. Corré el producer y 2 consumers en el mismo group. Confirmá reparto 2-2 de partitions.
3. Matá un consumer mid-run. Observá rebalancing en el otro (toma las 4).
4. Forzá lag: producer escribe 10× más rápido que consumer puede procesar. Usá `kafka-consumer-groups --describe` para ver lag.
5. Bonus: integrá con Flink o Spark Structured Streaming para hacer aggregations rolling.

## Conclusiones

- Topic + partitions + offsets = el modelo mental fundamental; Kinesis/Pub/Sub son re-skins.
- Key → partition determinístico → orden por key garantizado, paralelismo entre keys.
- At-least-once + dedupe en DB resuelve el 95% de los casos sin pagar exactly-once overhead.
- Consumer lag es la métrica de salud — alertar si crece sin techo.

## ✅ Soluciones de los ejercicios

Kafka necesita un broker (JVM + Zookeeper/KRaft) que no está en el laboratorio. Construimos
un **mini-Kafka en memoria** que reproduce lo esencial: *topic* con N *partitions*,
*partitioner* por key, *offsets*, *consumer groups* con rebalancing, y la semántica
*at-least-once*. El código de `confluent-kafka` real queda como referencia en las celdas de
arriba. Todo corre sin broker ni internet.

### Ejercicio 1 — Topic `clicks` con 4 partitions

Un topic es un log particionado. Modelamos el topic como una lista de 4 logs (uno por
partition).

In [ ]:
from collections import defaultdict
import hashlib, json

class Topic:
    def __init__(self, name, num_partitions):
        self.name = name
        self.partitions = [[] for _ in range(num_partitions)]   # cada partition = log ordenado

    @property
    def num_partitions(self):
        return len(self.partitions)

clicks = Topic("clicks", num_partitions=4)
print("topic:", clicks.name, "| partitions:", clicks.num_partitions)
assert clicks.num_partitions == 4
assert all(p == [] for p in clicks.partitions)
print("OK ejercicio 1 — topic 'clicks' con 4 partitions creado")

### Ejercicio 2 — Producer: misma `key` → misma partition

El *partitioner* por defecto de Kafka: `partition = hash(key) % num_partitions`. Así todos
los mensajes de un `user_id` caen SIEMPRE en la misma partition (ordenamiento por key).

In [ ]:
def partition_for(key, num_partitions):
    h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return h % num_partitions

def produce(topic, key, value):
    p = partition_for(key, topic.num_partitions)
    topic.partitions[p].append((key, value))
    return p

import random
random.seed(0)
placements = defaultdict(set)
for i in range(1000):
    uid = random.randint(1, 50)
    p = produce(clicks, key=uid, value={"page": "/foo", "ts": i})
    placements[uid].add(p)

total = sum(len(p) for p in clicks.partitions)
print("mensajes producidos:", total)
print("partition de user_id=7:", partition_for(7, 4), "(siempre la misma)")

assert total == 1000
assert all(len(parts) == 1 for parts in placements.values()), "cada user_id cae en 1 sola partition"
print("OK ejercicio 2 — misma key -> misma partition (orden por key garantizado)")

### Ejercicio 3 — Consumer 1 instancia, commit cada 100

El consumer lee secuencialmente y **commitea el offset** cada 100 mensajes. El offset marca
"hasta acá procesé": si reinicia, retoma desde ahí.

In [ ]:
class Consumer:
    def __init__(self, topic, assigned):
        self.topic = topic
        self.assigned = assigned              # partitions asignadas
        self.position = {p: 0 for p in assigned}
        self.committed = {p: 0 for p in assigned}

    def poll_all(self, commit_every=100):
        processed = 0
        for p in self.assigned:
            log = self.topic.partitions[p]
            while self.position[p] < len(log):
                self.position[p] += 1
                processed += 1
                if processed % commit_every == 0:
                    self.commit()
        self.commit()
        return processed

    def commit(self):
        self.committed = dict(self.position)

c = Consumer(clicks, assigned=[0, 1, 2, 3])
n = c.poll_all(commit_every=100)
print("procesados:", n, "| offsets committeados:", c.committed)

assert n == 1000
assert c.committed == c.position, "tras el commit final, committed == position"
print("OK ejercicio 3 — consumer único procesó las 4 partitions y committeó offsets")

### Ejercicio 4 — Consumer group de 2 instancias + rebalancing

Con 4 partitions y 2 consumers del mismo `group.id`, Kafka reparte 2-2. Si uno muere, el
otro toma las 4 (*rebalancing*). Modelamos la asignación tipo *range*.

In [ ]:
def assign_partitions(num_partitions, num_consumers):
    """Reparte partitions entre consumers (round-robin), como el group coordinator."""
    assignment = {c: [] for c in range(num_consumers)}
    for p in range(num_partitions):
        assignment[p % num_consumers].append(p)
    return assignment

two = assign_partitions(4, 2)
print("2 consumers:", two)
assert sorted(len(v) for v in two.values()) == [2, 2], "reparto 2-2"

# muere el consumer 1 -> rebalancing con 1 solo consumer
one = assign_partitions(4, 1)
print("tras matar uno:", one)
assert one[0] == [0, 1, 2, 3], "el sobreviviente toma las 4 partitions"
print("OK ejercicio 4 — consumer group reparte 2-2 y rebalancea a 4 al caer un consumer")

### Ejercicio 5 — At-least-once explícito

Con `enable.auto.commit=False`: procesás y **después** commiteás. Si el proceso crashea
entre "procesar" y "commit", al reiniciar se re-lee desde el último offset committeado →
**mensaje duplicado**. Esa es la garantía *at-least-once*.

In [ ]:
log = [("u1", {"ts": i}) for i in range(5)]     # una partition con 5 mensajes
delivered = []

def consume_until_crash(log, committed_offset, crash_at):
    pos = committed_offset
    while pos < len(log):
        delivered.append(log[pos][1]["ts"])      # PROCESAR (efecto colateral)
        if pos == crash_at:
            return pos                             # CRASH antes de commitear
        pos += 1                                   # commit implícito del avance
    return pos

# 1ª corrida: procesa 0,1,2 y crashea en 2 (antes de commitear el 2)
committed = consume_until_crash(log, committed_offset=0, crash_at=2)
print("entregados antes del crash:", delivered, "| offset committeado:", committed)

# reinicio: retoma desde el offset committeado (2) -> re-procesa el 2 = DUPLICADO
consume_until_crash(log, committed_offset=committed, crash_at=99)
print("entregados totales:", delivered)

assert delivered.count(2) == 2, "at-least-once: el mensaje 2 se entregó dos veces"
assert set(delivered) == {0, 1, 2, 3, 4}, "ningún mensaje se perdió"
print("OK ejercicio 5 — at-least-once: sin pérdida, pero con posible duplicado")